# Imports

In [3]:
import os
import math
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
import torch

# Config / Paths 

In [4]:
MODEL_NAME = "google/flan-t5-base"   # base model to adapt
OUTPUT_DIR = "/content/sample_data/flan_t5_lora_medquad"
CSV_PATH = "/content/sample_data/final_dataset_cleaned.csv"
EPOCHS = 8
BATCH_SIZE = 4                       # reduce if OOM (try 4)
LEARNING_RATE = 2e-4
MAX_SOURCE_LEN = 512
MAX_TARGET_LEN = 528
GRAD_ACCUM = 2                        # increase to simulate larger batch
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.manual_seed(SEED)

ds = load_dataset("csv", data_files={"train": CSV_PATH})

# Tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Preprocessing

In [6]:
def preprocess(batch):
    # add an instruction prefix to help model
    inputs = [f"Question: {q}\nAnswer:" for q in batch["question"]]
    targets = batch["answer"]
    model_inputs = tokenizer(inputs, max_length=MAX_SOURCE_LEN, truncation=True, padding="max_length")

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=MAX_TARGET_LEN, truncation=True, padding="max_length")
    # replace padding token id's in labels by -100 so they are ignored by loss
    label_ids = labels["input_ids"]
    label_ids = [[(tid if tid != tokenizer.pad_token_id else -100) for tid in seq] for seq in label_ids]

    model_inputs["labels"] = label_ids
    return model_inputs

tokenized = ds["train"].map(preprocess, batched=True, remove_columns=ds["train"].column_names)

# Load model in 8-bit and prepare for k-bit training

In [ ]:
# Define quantization config
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,         # or load_in_4bit=True for 4-bit
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)

# Load model with quantization
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

# Prepares model for k-bit training (needed when using int8)
model = prepare_model_for_kbit_training(model)

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

# Configure LoRA

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"],  # attention layers
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

model = get_peft_model(model, lora_config)

# ---------- Data collator ----------
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

# Training args & Trainer

In [9]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=True,                 # use fp16 mixed precision if GPU supports it
    logging_steps=50,
    save_total_limit=2,
    save_strategy="epoch",
    remove_unused_columns=True,
    push_to_hub=False,
    report_to=[], # set True if you want to push to HF hub
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

/tmp/ipython-input-3732259003.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# Train

In [10]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:186: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
50,4.587300
100,4.600100
150,4.738300
200,4.680000
250,4.592900
300,4.723300
350,4.631800
400,4.640300
450,4.602000
500,4.754100


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:186: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reen

TrainOutput(global_step=1000, training_loss=4.641633438110351, metrics={'train_runtime': 2424.5891, 'train_samples_per_second': 3.3, 'train_steps_per_second': 0.412, 'total_flos': 5521545363456000.0, 'train_loss': 4.641633438110351, 'epoch': 8.0})

# Save LoRA weights (and tokenizer)

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Done. LoRA adapters + tokenizer saved to:", OUTPUT_DIR)

Done. LoRA adapters + tokenizer saved to: /content/sample_data/flan_t5_lora_medquad
